## Import Packages

In [22]:
!pip install dask

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [23]:
import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [24]:
# paths for the three dfs

df_jan_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-01.csv"
df_feb_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-02.csv"
df_mar_path = "/kaggle/input/datasets/elemento/nyc-yellow-taxi-trip-data/yellow_tripdata_2016-03.csv"


- Based on the exploratory data analysis (EDA) of NYC Yellow Taxi trips, we identified outliers in the data. I selected specific columns that are useful for demand prediction, and these columns also contain outliers. Therefore, I need to remove the outliers from these specific columns.
- Columns are 'trip_distance', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'

In [25]:
# load the dataframes

df_jan = dd.read_csv(df_jan_path, assume_missing=True, usecols= ['trip_distance', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'])
df_feb = dd.read_csv(df_feb_path, assume_missing=True, usecols= ['trip_distance', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'])
df_mar = dd.read_csv(df_mar_path, assume_missing=True, usecols= ['trip_distance', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'])

In [26]:
# concat the three dataframes as one

df_final = dd.concat([df_jan, df_feb, df_mar], axis=0) # axis =0 means It works vertically (up and down).

In [105]:
df_final.head()

,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount
0,1.10,-73.990372,40.734695,-73.981842,40.732407,7.5
1,4.90,-73.980782,40.729912,-73.944473,40.716679,18.0
2,10.54,-73.984550,40.679565,-73.950272,40.788925,33.0
3,4.75,-73.993469,40.718990,-73.962242,40.657333,16.5
4,1.76,-73.960625,40.781330,-73.977264,40.758514,8.0


### **New york bounding box:**
min_latitude = 40.60
max_latitude = 40.85
min_longitude = -74.05
max_longitude = -73.70

In [27]:
# set the values of coordinates

min_latitude = 40.60
max_latitude = 40.85
min_longitude = -74.05
max_longitude = -73.70

In [28]:
# fare amount column
fare_amount = df_final["fare_amount"].compute()

# trip distance column
trip_distance = df_final["trip_distance"].compute()

In [29]:
fare_amount.shape[0]/10000000

3.4499859

In [30]:
## Percentile of fare amount
percentiles=np.arange(0.991,1,0.001)
fare_amount.quantile(percentiles)

0.991        52.00
0.992        52.00
0.993        52.00
0.994        52.00
0.995        54.00
0.996        58.50
0.997        63.00
0.998        69.00
0.999        81.00
1.000    429496.72
Name: fare_amount, dtype: float64

In [31]:
max_fare_amount_val = fare_amount.quantile(percentiles).iloc[-2].item()
min_fare_amount_val = 0.50

print(min_fare_amount_val)
print(max_fare_amount_val)

0.5
81.0


In [32]:
trip_distance.quantile(percentiles)

0.991          18.80
0.992          19.00
0.993          19.30
0.994          19.63
0.995          20.04
0.996          20.51
0.997          21.10
0.998          21.90
0.999          24.43
1.000    19072628.80
Name: trip_distance, dtype: float64

In [33]:
# percentile values for trip_distance

min_trip_distance_val = 0.25
max_trip_distance_val = trip_distance.quantile(percentiles).iloc[-2].item()

print(min_trip_distance_val)
print(max_trip_distance_val)

0.25
24.43


## Remove Outlier from the location data

In [34]:
# select data points within the given ranges

df_final = df_final.loc[(df_final["pickup_latitude"].between(min_latitude, max_latitude, inclusive="both")) & 
(df_final["pickup_longitude"].between(min_longitude, max_longitude, inclusive="both")) & 
(df_final["dropoff_latitude"].between(min_latitude, max_latitude, inclusive="both")) & 
(df_final["dropoff_longitude"].between(min_longitude, max_longitude, inclusive="both")), :]

## Remove Outliers from the Fare Amount data and Distance

In [35]:
df_final = df_final.loc[(df_final["fare_amount"].between(min_fare_amount_val,max_fare_amount_val,inclusive="both")) & 
(df_final["trip_distance"].between(min_trip_distance_val,max_trip_distance_val,inclusive="both"))]

## Remove Outliers from the Distance data

In [36]:
df_final = df_final.loc[(df_final["fare_amount"].between(min_fare_amount_val,max_fare_amount_val,inclusive="both")) & 
(df_final["trip_distance"].between(min_trip_distance_val,max_trip_distance_val,inclusive="both"))]

## Save After Removing Outlier

In [52]:
# save the pickup coordinates dataset

dir_path = "/kaggle/working/data/interim/"
file_path = os.path.join(dir_path, "location_data.csv")
pickup_coord_dataset = df_final.loc[:,['pickup_latitude','pickup_longitude']]

In [53]:
# form the dataset

pickup_coord_dataset = df_final.loc[:,['pickup_latitude','pickup_longitude']].compute()

print("Shape of the dataset is ", pickup_coord_dataset.shape)

Shape of the dataset is  (33234199, 2)


In [54]:
import os
os.makedirs(dir_path, exist_ok=True)

In [55]:
pickup_coord_dataset.to_csv(file_path, index=False)

## Making The Regions 

In [71]:
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

In [72]:
df_reader=pd.read_csv("/kaggle/working/data/interim/location_data.csv",chunksize=100000, usecols=["pickup_latitude","pickup_longitude"])

In [81]:
# 1. Re-initialize the iterator (since cell 73 consumed it completely)
df_reader = pd.read_csv("/kaggle/working/data/interim/location_data.csv", chunksize=100000, usecols=["pickup_latitude", "pickup_longitude"])

# 2. Train the MiniBatchKMeans model
mini_batch = MiniBatchKMeans(n_clusters=30, n_init=10, random_state=42)
for chunk in df_reader:
    scaled_chunk = scaler.transform(chunk)
    mini_batch.partial_fit(scaled_chunk)

# 3. Access centroids
mini_batch.cluster_centers_

array([[ 1.94149572,  0.6614905 ],
       [-0.13980742, -0.07596928],
       [-1.99721801,  1.44528196],
       [-3.83635934,  5.15118517],
       [-1.16343866, -0.81227839],
       [ 0.42137614, -0.13488625],
       [ 0.72209007,  2.8610836 ],
       [-0.52924948, -0.39015448],
       [ 1.07003845,  0.56520318],
       [-2.24451716, -0.3303071 ],
       [-1.00166487, -0.4036074 ],
       [ 1.16185672, -0.10175947],
       [-0.08902033, -0.55920139],
       [ 0.31820801,  1.59046972],
       [-0.04552948, -0.25372558],
       [ 0.20846098, -0.35389222],
       [-1.31780812,  0.52314378],
       [ 2.79102906,  0.81932304],
       [ 0.28712745,  0.12935608],
       [ 0.73099844, -0.27639436],
       [-1.53772972, -1.0082658 ],
       [ 0.67692302,  0.39844824],
       [-0.76412025, -0.74701446],
       [ 1.62394033,  0.1378545 ],
       [-3.07963657, -0.43385234],
       [ 0.3506753 , -0.54018812],
       [-0.57100818, -0.17842182],
       [-0.38830582, -0.76072368],
       [-2.66479788,

In [84]:
scaler.inverse_transform(mini_batch.cluster_centers_)

array([[ 40.80392392, -73.94975046],
       [ 40.74726528, -73.97685385],
       [ 40.69670159, -73.92094427],
       [ 40.64663525, -73.78474354],
       [ 40.7193993 , -74.00391496],
       [ 40.7625422 , -73.97901919],
       [ 40.77072843, -73.8689102 ],
       [ 40.73666362, -73.9884009 ],
       [ 40.78020052, -73.95328925],
       [ 40.68996945, -73.98620137],
       [ 40.72380321, -73.98889533],
       [ 40.78270006, -73.9778017 ],
       [ 40.74864784, -73.99461378],
       [ 40.75973368, -73.91560827],
       [ 40.74983178, -73.98338682],
       [ 40.75674608, -73.98706818],
       [ 40.71519695, -73.95483503],
       [ 40.82705049, -73.94394974],
       [ 40.75888759, -73.96930766],
       [ 40.77097094, -73.98421995],
       [ 40.70921009, -74.01111796],
       [ 40.76949887, -73.95941789],
       [ 40.73026981, -74.00151635],
       [ 40.79527921, -73.96899532],
       [ 40.66723527, -73.9900069 ],
       [ 40.76061753, -73.993915  ],
       [ 40.73552684, -73.98061923],
 

In [97]:
final_df=pd.read_csv("/kaggle/working/data/interim/location_data.csv")

In [98]:
final_df.head()

,pickup_latitude,pickup_longitude
0,40.734695,-73.990372
1,40.729912,-73.980782
2,40.679565,-73.984550
3,40.718990,-73.993469
4,40.781330,-73.960625


In [99]:
# prediction 
scaled_location_subset = scaler.transform(final_df)

scaled_location_subset

# get the cluster predictions

cluster_predictions = mini_batch.predict(scaled_location_subset)

cluster_predictions.shape

(33234199,)

In [101]:
# save the cluster predictions in data

# Save the cluster predictions directly into your Pandas DataFrame
final_df['region'] = cluster_predictions
final_df.head()

,pickup_latitude,pickup_longitude,region
0,40.734695,-73.990372,7
1,40.729912,-73.980782,26
2,40.679565,-73.984550,9
3,40.718990,-73.993469,10
4,40.781330,-73.960625,8


In [ ]:
# drop the latitude and logitude columns from data

time_series_data = df_final.drop(columns=["pickup_latitude","pickup_longitude"])

time_series_data